# Lab 01 — Fleet inventory (Assets pane)

**Control Plane pane:** _Assets_

**Zava context:** the platform team needs a single view of every agent, model deployment, evaluation, and connection Zava is running — so they can answer "who owns this, what does it call, and is it approved?"

In this lab you'll:

1. List **Foundry accounts and projects** in the subscription.
2. List **model deployments** for a project.
3. List **agents** created inside the project.
4. Enumerate **connections** (search, storage, App Insights).
5. Print a governance-friendly inventory table.

> Reference: [Manage agents across the fleet](https://learn.microsoft.com/en-us/azure/foundry/control-plane/how-to-manage-agents)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from rich.console import Console
from rich.table import Table

load_dotenv(Path.cwd().parent / ".env")
credential = DefaultAzureCredential()
console = Console()

SUB_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
RG = os.environ["AZURE_RESOURCE_GROUP"]
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

## 1. Foundry accounts (`Microsoft.CognitiveServices/accounts` of kind AIServices)

Foundry projects live inside a Foundry (AI Services) account. We use the management-plane SDK to enumerate them at subscription scope.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

cs_client = CognitiveServicesManagementClient(credential, SUB_ID)

accounts = [
    a for a in cs_client.accounts.list()
    if (a.kind or "").lower() in {"aiservices", "openai"}
]

table = Table(title="Foundry / AI accounts in subscription")
for col in ("Name", "Kind", "Location", "Resource group"):
    table.add_column(col)

for a in accounts:
    rg = a.id.split("/resourceGroups/")[1].split("/")[0]
    table.add_row(a.name, a.kind or "", a.location, rg)
console.print(table)
print(f"Total: {len(accounts)}")

## 2. Model deployments in the project

In [ ]:
table = Table(title="Deployments in the current Foundry project")
for col in ("Name", "Model", "Type", "SKU / Capacity"):
    table.add_column(col)

for d in project.deployments.list():
    table.add_row(
        getattr(d, "name", ""),
        getattr(d, "model_name", getattr(d, "model", "")),
        getattr(d, "type", getattr(d, "deployment_type", "")),
        str(getattr(d, "sku", getattr(d, "capacity", ""))),
    )
console.print(table)

## 3. Agents in the project

For the labs to have something to list, we'll register a small Zava agent if none exist. Rerun the cell — it's idempotent.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

MODEL = os.environ["FOUNDRY_MODEL_NAME"]

ZAVA_AGENTS = [
    {
        "name": "zava-support-bot",
        "instructions": (
            "You are the Zava support bot for an online home & garden retailer. "
            "Answer questions about orders, returns, and shipping using ONLY the return policy "
            "and product catalog provided to you. If you don't know, say so and offer to escalate."
        ),
    },
    {
        "name": "zava-product-advisor",
        "instructions": (
            "You are Zava's product advisor. Recommend home & garden products from the Zava catalog. "
            "Ask 1-2 clarifying questions if the customer is vague. Never invent products."
        ),
    },
]

existing = {a.name: a for a in project.agents.list()}
for spec in ZAVA_AGENTS:
    if spec["name"] in existing:
        print(f"[skip] {spec['name']} already exists (id={existing[spec['name']].id})")
        continue
    agent = project.agents.create_version(
        agent_name=spec["name"],
        definition=PromptAgentDefinition(
            model=MODEL,
            instructions=spec["instructions"],
        ),
    )
    print(f"[created] {agent.name} -> {agent.id}")

In [ ]:
table = Table(title="Agents in the current Foundry project")
for col in ("Name", "Id", "Model", "Created"):
    table.add_column(col)

for agent in project.agents.list():
    latest = agent.versions.latest
    table.add_row(
        agent.name,
        agent.id,
        getattr(latest.definition, "model", ""),
        str(latest.created_at),
    )
console.print(table)

## 4. Connections (search indexes, storage, App Insights…)

Connections are the external resources Zava's agents can call. From a governance perspective, every connection is a potential data exit — enumerate them so you know what to review.

In [ ]:
table = Table(title="Project connections")
for col in ("Name", "Type", "Target"):
    table.add_column(col)

for c in project.connections.list():
    table.add_row(
        getattr(c, "name", ""),
        str(getattr(c, "type", "")),
        str(getattr(c, "target", getattr(c, "endpoint_url", ""))),
    )
console.print(table)

## 5. Governance summary

Every fleet inventory report should answer three questions:

1. **What is running?** (agents + models)
2. **What does it depend on?** (connections)
3. **What is protecting it?** (guardrails, RBAC, tracing — coming in later labs)

Now check the same view in the portal:

> **Foundry portal → Operate → Assets**

You'll see Agents, Evaluations, Playgrounds, Connections and more, filterable by project. The **Assets** pane is what most SecOps and platform reviewers will use day-to-day.